# Week 03 — Data Visualisation: Matplotlib & Seaborn

Solutions for `23CSE301_ML_Week3_Guide.pdf`. Uses the standard Iris & Wine datasets.

**Iris** powers every example; **Wine** powers every exercise. Saved figures go to `figures/`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine

iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df['species'] = [iris.target_names[t] for t in iris.target]

wine = load_wine()
wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
wine_df['wine_class'] = [wine.target_names[t] for t in wine.target]

sns.set_theme(style='whitegrid')
colors = ['steelblue', 'tomato', 'seagreen']
wine_features = list(wine.feature_names)      # the 13 numeric features
wine_classes = list(wine.target_names)

# Part I — Matplotlib

## 1.1 Figure Anatomy and the Object-Oriented API

In [ ]:
# 1-2. y = x^2 over [-3, 3]; 3. save to figures/parabola.png
import os
os.makedirs('figures', exist_ok=True)  # ensure the folder exists on a fresh clone / CI
x = np.linspace(-3, 3, 50)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, x**2, color='steelblue', linewidth=2, label='$y = x^2$')
ax.set_title('The parabola y = x squared')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend()
plt.tight_layout()
plt.savefig('figures/parabola.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved figures/parabola.png')

## 1.2 Line Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
styles = ['-', '--', '-.']
for i, cls in enumerate(wine_classes):
    grp = wine_df[wine_df['wine_class'] == cls]
    ax.plot(grp.index, grp['alcohol'], color=colors[i], alpha=0.7,
            linestyle=styles[i], label=cls)  # class 2 dashed, class 3 dash-dot
ax.axhline(wine_df['alcohol'].mean(), color='black', linestyle=':', label='overall mean')
ax.set_xlabel('Sample index'); ax.set_ylabel('Alcohol')
ax.set_title('Alcohol across samples by wine class')
ax.legend()
plt.tight_layout(); plt.show()

## 1.3 Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# 1-2. alcohol vs color_intensity, coloured by class, s=60, white edges
for i, cls in enumerate(wine_classes):
    m = wine_df['wine_class'] == cls
    axes[0].scatter(wine_df.loc[m, 'alcohol'], wine_df.loc[m, 'color_intensity'],
                    color=colors[i], s=60, edgecolors='white', alpha=0.8, label=cls)
axes[0].set(xlabel='Alcohol', ylabel='Color intensity', title='Alcohol vs color_intensity')
axes[0].legend()
# 3. flavanoids vs proline -- visibly more linearly separable by class
for i, cls in enumerate(wine_classes):
    m = wine_df['wine_class'] == cls
    axes[1].scatter(wine_df.loc[m, 'flavanoids'], wine_df.loc[m, 'proline'],
                    color=colors[i], s=60, edgecolors='white', alpha=0.8, label=cls)
axes[1].set(xlabel='Flavanoids', ylabel='Proline', title='Flavanoids vs proline')
axes[1].legend()
plt.tight_layout(); plt.show()

## 1.4 Histograms

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for i, cls in enumerate(wine_classes):
    data = wine_df.loc[wine_df['wine_class'] == cls, 'proline']
    ax.hist(data, bins=15, color=colors[i], alpha=0.6, density=True,
            label=cls, edgecolor='white')                  # 2. density=True
    ax.axvline(data.mean(), color=colors[i], linestyle='--')  # 3. per-class mean line
ax.set_xlabel('Proline'); ax.set_ylabel('Density')
ax.set_title('Proline distribution by class (class_0 has the highest proline)')
ax.legend()
plt.tight_layout(); plt.show()

## 1.5 Bar Charts

In [ ]:
means = wine_df.groupby('wine_class')['alcohol'].mean().reindex(wine_classes)
stds = wine_df.groupby('wine_class')['alcohol'].std().reindex(wine_classes)
counts = wine_df['wine_class'].value_counts().reindex(wine_classes)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(wine_classes))
# 1. mean alcohol +/- 1 SD
axes[0].bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(wine_classes)
axes[0].set_title('Mean alcohol +/- 1 SD'); axes[0].set_ylabel('Alcohol')
# 2. class counts (mild imbalance: 59 / 71 / 48)
axes[1].bar(wine_classes, counts.values, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_title('Sample count per class (slightly imbalanced)'); axes[1].set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# 3. grouped bar: mean flavanoids vs mean total_phenols per class
fmean = wine_df.groupby('wine_class')['flavanoids'].mean().reindex(wine_classes)
pmean = wine_df.groupby('wine_class')['total_phenols'].mean().reindex(wine_classes)
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(wine_classes))
ax.bar(x - 0.2, fmean, width=0.4, label='flavanoids', color='steelblue', edgecolor='black')
ax.bar(x + 0.2, pmean, width=0.4, label='total_phenols', color='tomato', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(wine_classes)
ax.set_ylabel('Mean value'); ax.set_title('Flavanoids vs total_phenols per class')
ax.legend()
plt.tight_layout(); plt.show()

## 1.6 Box Plots

In [ ]:
def boxes_by_class(ax, feat):
    data = [wine_df.loc[wine_df['wine_class'] == c, feat].values for c in wine_classes]
    bp = ax.boxplot(data, tick_labels=wine_classes, patch_artist=True)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set_title(feat)

# 1-2. color_intensity vs hue (hue separates classes more cleanly)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
boxes_by_class(axes[0], 'color_intensity')
boxes_by_class(axes[1], 'hue')
plt.tight_layout(); plt.show()

In [ ]:
# 3. (1, 4) grid of box plots across classes
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, feat in zip(axes, ['alcohol', 'flavanoids', 'color_intensity', 'proline']):
    boxes_by_class(ax, feat)
plt.tight_layout(); plt.show()

## 1.7 Multi-Panel Subplots

In [ ]:
# 1. 3x5 grid of histograms of all 13 wine features, coloured by class
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
for ax, feat in zip(axes.ravel(), wine_features):
    for i, cls in enumerate(wine_classes):
        data = wine_df.loc[wine_df['wine_class'] == cls, feat]
        ax.hist(data, bins=12, color=colors[i], alpha=0.6, label=cls)
    ax.set_title(feat, fontsize=8)
for ax in axes.ravel()[len(wine_features):]:  # blank the 2 unused panels
    ax.axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', title='wine_class')
fig.suptitle('All Wine feature distributions by class', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# 2. same grid, but scatter of alcohol vs each feature
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
for ax, feat in zip(axes.ravel(), wine_features):
    for i, cls in enumerate(wine_classes):
        m = wine_df['wine_class'] == cls
        ax.scatter(wine_df.loc[m, 'alcohol'], wine_df.loc[m, feat],
                   color=colors[i], s=10, alpha=0.6)
    ax.set_title(feat, fontsize=8)
for ax in axes.ravel()[len(wine_features):]:
    ax.axis('off')
# 3. manual spacing instead of tight_layout
fig.suptitle('Alcohol vs each feature by class', fontsize=14)
plt.subplots_adjust(hspace=0.5, wspace=0.4)
plt.show()

# Part II — Seaborn

## 2.1 Scatter Plot with Hue

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
# 1-2. hue + style by class; 3. Set2 palette is readable & colourblind-friendlier
sns.scatterplot(data=wine_df, x='alcohol', y='color_intensity',
                hue='wine_class', style='wine_class', palette='Set2', alpha=0.85, ax=ax)
ax.set_title('Alcohol vs color_intensity by class (palette=Set2)')
plt.tight_layout(); plt.show()
# Set2 is muted and qualitatively distinct, so classes stay separable for colourblind viewers.

## 2.2 Distribution Plots (histplot & kdeplot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 1. histplot with kde (class_0 has the widest proline spread)
sns.histplot(data=wine_df, x='proline', hue='wine_class', kde=True, bins=20, alpha=0.5, ax=axes[0])
axes[0].set_title('Histogram + KDE: proline')
# 2-3. kdeplot fill -- cleaner for overlapping comparison
sns.kdeplot(data=wine_df, x='proline', hue='wine_class', fill=True, alpha=0.3, ax=axes[1])
axes[1].set_title('KDE: proline (cleaner for overlap)')
plt.tight_layout(); plt.show()

## 2.3 Box Plots and Violin Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 1. box vs violin for color_intensity
sns.boxplot(data=wine_df, x='wine_class', y='color_intensity',
            hue='wine_class', palette='tab10', legend=False, ax=axes[0])
axes[0].set_title('Box: color_intensity')
# 3. inner='stick' shows individual points
sns.violinplot(data=wine_df, x='wine_class', y='color_intensity',
               hue='wine_class', palette='tab10', legend=False, inner='stick', ax=axes[1])
axes[1].set_title('Violin (inner=stick): color_intensity')
plt.tight_layout(); plt.show()
# 2. The violin hints at a longer/second mode in class_2 that the box plot compresses into fliers.

## 2.4 Pair Plots

In [ ]:
# 2. restricted pairplot -- flavanoids & proline give the cleanest separation
g = sns.pairplot(wine_df, hue='wine_class', palette='tab10', diag_kind='kde',
                 vars=['alcohol', 'color_intensity', 'flavanoids', 'proline'],
                 plot_kws={'alpha': 0.6})
g.figure.suptitle('Wine pairwise relationships (4 key features)', y=1.02)
plt.show()

## 2.5 Heatmaps

In [ ]:
corr = wine_df[wine_features].corr()
# 1 + 2. lower-triangle correlation heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 7}, ax=ax)
ax.set_title('Wine feature correlations (flavanoids & total_phenols most positive)')
plt.tight_layout(); plt.show()

In [ ]:
# 3. per-class mean heatmap (3 x 13)
class_means = wine_df.groupby('wine_class')[wine_features].mean()
fig, ax = plt.subplots(figsize=(13, 3))
sns.heatmap(class_means, annot=True, fmt='.1f', cmap='viridis', annot_kws={'size': 7}, ax=ax)
ax.set_title('Per-class feature means')
plt.tight_layout(); plt.show()

## 2.6 Bar and Count Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 1. mean proline per class with SD error bars
sns.barplot(data=wine_df, x='wine_class', y='proline', hue='wine_class',
            palette='tab10', legend=False, errorbar='sd', ax=axes[0])
axes[0].set_title('Mean proline per class (+/- SD)')
# 2. class balance (slightly imbalanced)
sns.countplot(data=wine_df, x='wine_class', hue='wine_class',
              palette='tab10', legend=False, ax=axes[1])
axes[1].set_title('Class balance')
plt.tight_layout(); plt.show()

In [ ]:
# 3. grouped bar via melt: mean alcohol vs color_intensity per class
melted = pd.melt(wine_df, id_vars='wine_class', value_vars=['alcohol', 'color_intensity'])
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=melted, x='wine_class', y='value', hue='variable', errorbar='sd', ax=ax)
ax.set_title('Mean alcohol vs color_intensity per class')
plt.tight_layout(); plt.show()

## 2.7 FacetGrid — Small Multiples

In [ ]:
# 1. histplot of alcohol per class
g = sns.FacetGrid(wine_df, col='wine_class', height=3.2, aspect=0.9)
g.map(sns.histplot, 'alcohol', bins=12, kde=True, color='steelblue')
g.set_titles(col_template='{col_name}')
g.figure.suptitle('Alcohol distribution per class', y=1.03)
plt.show()

In [ ]:
# 3. add a row dimension by binning total_phenols -> 2 x 3 grid of scatters
wine_df['high_phenols'] = pd.cut(wine_df['total_phenols'], bins=2, labels=['low', 'high'])
g = sns.FacetGrid(wine_df, row='high_phenols', col='wine_class', height=3, aspect=1.0)
g.map(sns.scatterplot, 'color_intensity', 'alcohol', alpha=0.7)
g.figure.suptitle('color_intensity vs alcohol by class and phenol level', y=1.03)
plt.show()

# Part III — Capstone: Complete Visual EDA of the Wine Dataset

Exactly eight figures, in order, each titled and labelled.

In [ ]:
# Figure 1 — Class balance
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=wine_df, x='wine_class', hue='wine_class', palette='tab10', legend=False, ax=ax)
ax.set_title('Figure 1 - Class balance'); ax.set_xlabel('Wine class'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 2 — Feature overview (3x5 histograms)
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
for ax, feat in zip(axes.ravel(), wine_features):
    for i, cls in enumerate(wine_classes):
        ax.hist(wine_df.loc[wine_df['wine_class'] == cls, feat], bins=12, color=colors[i], alpha=0.6)
    ax.set_title(feat, fontsize=8)
for ax in axes.ravel()[len(wine_features):]:
    ax.axis('off')
fig.suptitle('Figure 2 - Feature overview by class', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 3 — Best separators (flavanoids vs proline)
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=wine_df, x='flavanoids', y='proline', hue='wine_class',
                style='wine_class', palette='tab10', s=60, ax=ax)
ax.set_title('Figure 3 - Best separators: flavanoids vs proline')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 4 — Distribution comparison (proline & alcohol KDEs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=wine_df, x='proline', hue='wine_class', fill=True, alpha=0.3, ax=axes[0])
axes[0].set_title('Proline by class')
sns.kdeplot(data=wine_df, x='alcohol', hue='wine_class', fill=True, alpha=0.3, ax=axes[1])
axes[1].set_title('Alcohol by class')
fig.suptitle('Figure 4 - Distribution comparison', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 5 — Box vs violin (color_intensity)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=wine_df, x='wine_class', y='color_intensity', hue='wine_class',
            palette='tab10', legend=False, ax=axes[0])
axes[0].set_title('Box')
sns.violinplot(data=wine_df, x='wine_class', y='color_intensity', hue='wine_class',
               palette='tab10', legend=False, inner='quartile', ax=axes[1])
axes[1].set_title('Violin')
fig.suptitle('Figure 5 - color_intensity: box vs violin', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 6 — Grouped bar (flavanoids, total_phenols, proanthocyanins)
feats6 = ['flavanoids', 'total_phenols', 'proanthocyanins']
melted = pd.melt(wine_df, id_vars='wine_class', value_vars=feats6)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=melted, x='wine_class', y='value', hue='variable', errorbar='sd', ax=ax)
ax.set_title('Figure 6 - Mean phenolic features per class (+/- SD)')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 7 — Lower-triangle correlation heatmap
corr = wine_df[wine_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 7}, ax=ax)
ax.set_title('Figure 7 - Correlation heatmap')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 8 — Pairplot of the 4 most discriminating features
g = sns.pairplot(wine_df, hue='wine_class', palette='tab10', diag_kind='kde',
                 vars=['flavanoids', 'proline', 'color_intensity', 'alcohol'],
                 plot_kws={'alpha': 0.6})
g.figure.suptitle('Figure 8 - Pairplot of key features', y=1.02)
plt.show()

### Capstone summary

The Wine dataset separates cleanly on the phenolic features: **flavanoids** and **proline** are the strongest single-feature class separators, and the `flavanoids` vs `proline` scatter (Figure 3) shows the three classes forming distinct clusters. The classes are only mildly imbalanced (roughly 59 / 71 / 48 samples), so no aggressive resampling is needed before modelling. The correlation heatmap (Figure 7) shows the strongest positive correlation between **flavanoids** and **total_phenols**, meaning they carry overlapping information and one could be dropped. `color_intensity` is right-skewed with a longer tail in one class (visible in the violin plot), and `proline` has by far the widest per-class spread. Overall, a handful of phenolic/colour features already separate the classes well, which bodes well for a simple classifier.